# Can we predict late deliveries?
Protocol: `protocol.yaml` (temporal holdout, 5-split walk-forward CV, choose by mean CV ROC AUC, score holdout once).

Feature decision: `claim_paid_eur` is compensation on a claim *settled for the consignment*, i.e. recorded after delivery, and it nearly determines the target (about 90% late when >0, never late when 0). It is not known at prediction time, so it is excluded. `consignment` (identifier) and `dispatched_on` (split only) are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
df=pd.read_csv('data.csv'); df['dispatched_on']=pd.to_datetime(df['dispatched_on'])
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==8051 and len(hold_idx)==1610
train=df.drop(index=hold_idx).sort_values('dispatched_on',kind='stable')  # original index kept = 0-based row position
hold=df.loc[np.sort(hold_idx)]
assert train.dispatched_on.max()<=hold.dispatched_on.min()
print(len(train),len(hold),train.late_delivery.mean(),hold.late_delivery.mean())

In [ ]:
TARGET='late_delivery'
NUM=['distance_km','weight_kg','parcels']; CAT=['carrier']
FEATURES=NUM+CAT   # claim_paid_eur excluded: recorded after the outcome
X,y=train[FEATURES],train[TARGET]

Candidates are named before any is scored.

In [ ]:
def prep(): return ColumnTransformer([('n',StandardScaler(),NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
CANDIDATES={
 'baseline_prior':DummyClassifier(strategy='prior'),
 'logreg_C0.1':make_pipeline(prep(),LogisticRegression(C=0.1,max_iter=1000)),
 'logreg_C1':make_pipeline(prep(),LogisticRegression(C=1.0,max_iter=1000)),
 'hgb_depth3':make_pipeline(prep(),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)),
 'hgb_depth6':make_pipeline(prep(),HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=200,random_state=0)),
}
print(list(CANDIDATES))

In [ ]:
tscv=TimeSeriesSplit(n_splits=5)
folds=[(tr,va) for tr,va in tscv.split(X)]   # positions in time-sorted training rows
cv_scores={}
for name,m in CANDIDATES.items():
    s=[]
    for tr,va in folds:
        mm=clone(m).fit(X.iloc[tr],y.iloc[tr])   # preprocessing is inside the pipeline, fit per fold
        s.append(roc_auc_score(y.iloc[va],mm.predict_proba(X.iloc[va])[:,1]))
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); print(cv.round(4))
chosen=cv['mean'].idxmax(); print('chosen:',chosen)

In [ ]:
final=clone(CANDIDATES[chosen]).fit(X,y)
holdout_auc=roc_auc_score(hold[TARGET],final.predict_proba(hold[FEATURES])[:,1])   # scored once
print('holdout roc_auc',holdout_auc)

In [ ]:
idx=train.index.to_numpy()
ev={'features':FEATURES,'candidates':list(CANDIDATES),
 'cv_folds':[[[int(i) for i in idx[tr]],[int(i) for i in idx[va]]] for tr,va in folds],
 'cv_scores':{k:[float(x) for x in v] for k,v in cv_scores.items()},
 'chosen':chosen,'holdout_rows':[int(i) for i in hold.index],
 'holdout_scores':{chosen:float(holdout_auc)}}
json.dump(ev,open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. Without the post-outcome claim column, the remaining features (carrier, distance, weight, parcels) are the only legitimate predictors; the holdout AUC shows how much signal they carry.